# Milestone 4 — QLoRA fine-tuning (Gemma 4 E4B)

## Pre-flight checklist (do BEFORE starting H100)
1. **Push latest code to GitHub** — the clone cell pulls from GitHub; unpushed local fixes won't appear
2. **Create HF model repo** — e.g. `your-username/gemma4-icd-cpt-qlora` (Settings → New model)
3. **Set `HUB_MODEL_ID`** in the config cell below
4. **Runtime → H100 GPU** (A100/T4 also work; batch settings auto-adjust)
5. **HF write token** + **W&B API key** ready
6. Run all cells top → bottom without skipping

Outputs: LoRA adapter on HF Hub + W&B loss curves + local checkpoint

In [ ]:
!pip install -q 'transformers>=4.50.0' 'trl>=0.9.0' 'peft>=0.10.0' accelerate bitsandbytes datasets huggingface_hub wandb

In [ ]:
import os
REPO = '/content/clinical-code-extractor'
if not os.path.exists(REPO):
    !git clone https://github.com/namitrathod/clinical-code-extractor.git
else:
    %cd {REPO}
    !git pull --ff-only
%cd /content/clinical-code-extractor

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/content/clinical-code-extractor')
sys.path.insert(0, str(ROOT))

MODEL_ID = 'google/gemma-4-E4B-it'
HUB_MODEL_ID = 'your-username/gemma4-icd-cpt-qlora'  # <-- REQUIRED: change before run

WANDB_PROJECT = 'clinical-code-extractor'
WANDB_RUN_NAME = 'gemma4-icd-cpt-qlora'

NUM_EPOCHS = 2
LEARNING_RATE = 2e-4

# Auto-set batch/seq from GPU (override manually if needed)
import torch
if torch.cuda.is_available():
    _gpu = torch.cuda.get_device_name(0)
    if 'H100' in _gpu or 'A100' in _gpu:
        PER_DEVICE_BATCH, GRAD_ACCUM, MAX_SEQ_LENGTH = 8, 2, 2048
    else:
        PER_DEVICE_BATCH, GRAD_ACCUM, MAX_SEQ_LENGTH = 4, 4, 1024
    print('GPU:', _gpu)
else:
    PER_DEVICE_BATCH, GRAD_ACCUM, MAX_SEQ_LENGTH = 2, 8, 1024
    print('WARNING: no GPU detected — training will fail')

if HUB_MODEL_ID.startswith('your-username'):
    raise ValueError('Set HUB_MODEL_ID to your Hugging Face repo (e.g. namitrathod/gemma4-icd-cpt-qlora)')

print('Model:', MODEL_ID)
print('Hub target:', HUB_MODEL_ID)
print('Batch:', PER_DEVICE_BATCH, 'x accum', GRAD_ACCUM, '=', PER_DEVICE_BATCH * GRAD_ACCUM)
print('Max seq length:', MAX_SEQ_LENGTH)

In [ ]:
from huggingface_hub import login, create_repo
import os
import wandb

login()  # HF write token
create_repo(HUB_MODEL_ID, repo_type='model', exist_ok=True)
print('HF repo ready:', HUB_MODEL_ID)

wandb.login()
os.environ['WANDB_PROJECT'] = WANDB_PROJECT

In [ ]:
!python scripts/prepare_training_data.py --spot-check 2

In [ ]:
import json
from datasets import load_dataset
from transformers import AutoTokenizer
from src.prompts import format_training_text, format_for_model

train_ds = load_dataset('json', data_files=str(ROOT / 'data/train_sft.jsonl'), split='train')
val_ds = load_dataset('json', data_files=str(ROOT / 'data/val_sft.jsonl'), split='train')
assert len(train_ds) >= 3000, f'Expected ~4000 train rows, got {len(train_ds)}'
assert len(val_ds) >= 400, f'Expected ~500 val rows, got {len(val_ds)}'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- pre-flight: token lengths + format sanity ---
lengths = []
for row in train_ds:
    text = format_training_text(tokenizer, row['note'], row['icd10'], row['cpt'])
    lengths.append(len(tokenizer(text, add_special_tokens=False)['input_ids']))
over = sum(1 for n in lengths if n > MAX_SEQ_LENGTH)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print(f'Tokens: max={max(lengths)}, p95={sorted(lengths)[int(0.95*len(lengths))]}, over limit={over}')
if over > len(train_ds) * 0.05:
    raise ValueError(f'{over} rows exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} — increase it')

sample = train_ds[0]
assert json.loads(sample['completion'])['icd10'] == sample['icd10']
infer_prefix = format_for_model(tokenizer, sample['note'], tokenize=False)
train_text = format_training_text(tokenizer, sample['note'], sample['icd10'], sample['cpt'])
assert infer_prefix in train_text, 'Inference prompt must match training user turn'
print('Sample completion:', sample['completion'])
print('Pre-flight OK')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules='all-linear',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer
from src.prompts import format_training_text

def formatting_func(examples):
    """Handle single row or batched rows from SFTTrainer."""
    if isinstance(examples['note'], list):
        return [
            format_training_text(tokenizer, n, i, c)
            for n, i, c in zip(examples['note'], examples['icd10'], examples['cpt'])
        ]
    return format_training_text(
        tokenizer, examples['note'], examples['icd10'], examples['cpt']
    )

training_args = SFTConfig(
    output_dir='checkpoints/gemma4-icd-cpt-qlora',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=True,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    assistant_only_loss=True,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_strategy='every_save',
    report_to='wandb',
    run_name=WANDB_RUN_NAME,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

print('Effective batch size:', PER_DEVICE_BATCH * GRAD_ACCUM)
print('Training steps (~):', len(train_ds) // (PER_DEVICE_BATCH * GRAD_ACCUM) * NUM_EPOCHS)

In [ ]:
trainer.train()
trainer.push_to_hub(commit_message='QLoRA clinical coding adapter')
print('Adapter pushed to:', HUB_MODEL_ID)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
Path('results').mkdir(exist_ok=True)

history = trainer.state.log_history
train_loss = [(h['step'], h['loss']) for h in history if 'loss' in h]
eval_loss = [(h['epoch'], h['eval_loss']) for h in history if 'eval_loss' in h]

if train_loss:
    plt.figure(figsize=(8, 4))
    plt.plot([x[0] for x in train_loss], [x[1] for x in train_loss], label='train')
    if eval_loss:
        plt.plot([x[0] for x in eval_loss], [x[1] for x in eval_loss], 'o-', label='eval')
    plt.xlabel('step / epoch')
    plt.ylabel('loss')
    plt.legend()
    plt.title('QLoRA training')
    plt.savefig('results/train_loss.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved results/train_loss.png')

## Smoke test — 3 held-out notes

Valid JSON with whitelisted codes → proceed to Milestone 5 eval.

In [ ]:
import json
from src.prompts import format_for_model
from src.validate import process_model_output, load_whitelists

# Reuse trained model (avoid loading a second copy into VRAM)
ft_model = trainer.model
ft_model.eval()

icd_w, cpt_w = load_whitelists(ROOT / 'codes')
test_rows = [json.loads(l) for l in open(ROOT / 'data/test.jsonl')][:3]

for row in test_rows:
    prompt = format_for_model(tokenizer, row['note'], tokenize=False)
    inputs = tokenizer(prompt, return_tensors='pt').to(ft_model.device)
    out = ft_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    raw = tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    result = process_model_output(raw, icd_w, cpt_w)
    print('---', row['id'], '---')
    print('Gold ICD:', row['icd10'], '| CPT:', row['cpt'])
    print('Pred ICD:', result['icd10'], '| CPT:', result['cpt'])
    print('JSON valid:', result['json_valid'])
    print('Raw:', raw[:200])